<a href="https://colab.research.google.com/github/mahdad277/repo1/blob/agentic-rag3/AgenticRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install langchain langchain-community
%pip install chromadb
%pip install -U :class:`~langchain-huggingface
%pip install langchain-google-genai

In [ ]:
# 1. Load your documents (e.g., from a PDF, text, or web)
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.ibm.com/policy")

docs = loader.load()

# 2. Split the documents into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

# 3. Create Vector Store and Retriever (using a free embedding model like all-MiniLM-L6-v2)
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Note: Running embeddings locally with HuggingFace is free!
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

# 4. Create the Tool for the Agent
# This turns the retriever into a callable function/tool.
from langchain.tools.retriever import create_retriever_tool
retrieval_tool = create_retriever_tool(
    retriever,
    name="company_policy_search",
    description="Tool for searching and retrieving information about the company's internal policies."
)

In [ ]:
# 1. Initialize the LLM (using the free Gemini API for best results)
# Replace 'YOUR_GEMINI_API_KEY' with your actual key or set it as an environment variable
# Note: You can also use Ollama for local models here.

from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
import os
from langchain_core.prompts import ChatPromptTemplate

# Ensure the environment variable is set or pass the key directly

api_key_value = userdata.get("GEMINI_API_KEY")

if api_key_value:
    print("API Key loaded successfully.")
    # You don't need to do anything else, as LangChain's ChatGoogleGenerativeAI
    # class automatically looks for the GEMINI_API_KEY environment variable.
    # If you needed to pass it manually, you would use:
    # llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=api_key_value)
else:
    print("ERROR: GEMINI_API_KEY environment variable not found!")
    # Exit or raise error here
    exit()
print(api_key_value)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=api_key_value,
    temperature=0.0
)

# 2. Define the Agent's Tools
tools = [retrieval_tool]

# 3. Create the Agent Executor (The brains of the operation)
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

# The prompt is critical: it tells the agent how to use the tool
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", """
You are a helpful assistant. Use the 'company_policy_search' tool ONLY if the user's question requires internal policy information.
If the question is general knowledge, answer directly.
If the question requires information from the policy, first use the tool to retrieve the context, then synthesize the final answer based on the retrieved information.
"""),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)


agent = create_tool_calling_agent(llm, tools, prompt_template)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 4. Run the Agent
response_1 = agent_executor.invoke({"input": "What is the new policy for taking personal days off?"})
print("\n--- Response 1 ---")
print(response_1["output"])

response_2 = agent_executor.invoke({"input": "What is the capital of France?"})
print("\n--- Response 2 ---")
print(response_2["output"])